# Experiment: Phase 3 WASP Behaviour Benchmark

This Kaggle notebook runs the repository's leakage-safe Phase 3 benchmark on a **private, attached** copy of the WASP-lab dataset. It reports public-dataset behaviour performance only; it does not validate the target collar, a farm, fever, lameness, diagnosis, or treatment.

Before running: attach the private WASP Dataset, enable Internet, create a Kaggle Secret named `WANDB_API_KEY`, and replace `REPOSITORY_COMMIT` with the immutable commit containing the Phase 3 code.

In [ ]:
from __future__ import annotations

import os
import random
import re
from pathlib import Path

SEED = 42
DATASET_DIR = Path('/kaggle/input/wasp-cow-imu')  # Set this to the attached private Dataset path.
KAGGLE_DATASET_REF = 'your-kaggle-handle/wasp-cow-imu:version'
OUTPUT_DIR = Path('/kaggle/working/phase3_wasp_benchmark')
REPOSITORY_URL = 'https://github.com/TaherPanbiharwala/Cattle-Fleet-Monitoring.git'
REPOSITORY_COMMIT = 'REPLACE_WITH_PHASE3_COMMIT_SHA'
WANDB_PROJECT = 'cattle-fleet-phase3'

random.seed(SEED)
if not re.fullmatch(r'[0-9a-fA-F]{40}', REPOSITORY_COMMIT):
    raise ValueError('REPOSITORY_COMMIT must be the full 40-character immutable Phase 3 Git SHA.')
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Attach the private WASP Kaggle Dataset; missing: {DATASET_DIR}')


## Reproducible setup

The notebook installs a pinned, public source revision. The private Kaggle input supplies raw data; it is never uploaded to W&B.

In [ ]:
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/Cattle-Fleet-Monitoring')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', REPOSITORY_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--quiet', '--tags', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', REPOSITORY_COMMIT], check=True)
RESOLVED_SOURCE_REVISION = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, text=True, capture_output=True,
).stdout.strip()
if RESOLVED_SOURCE_REVISION.casefold() != REPOSITORY_COMMIT.casefold():
    raise RuntimeError('Checked-out source revision does not match REPOSITORY_COMMIT.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', '.[ml]'], cwd=REPO_DIR, check=True)

# Kaggle Secret is read only into this process and is never printed or written to disk.
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = WANDB_PROJECT

print(f'Pinned source: {RESOLVED_SOURCE_REVISION}')
print(f'Input CSV root: {DATASET_DIR}')


## Pre-registered evaluation

- Input: MPU9250 acceleration and gyroscope axes only, plus derived magnitudes.
- Windows: 5 seconds at 10 Hz with 50% overlap; no event or timestamp gap may be crossed.
- Labels: Resting `0`, Grazing `1`, Walking `3`, and Miscellaneous `5` (Other/Unknown).
- Evaluation: leave-one-cow-out outer validation with cow-grouped inner selection.
- Candidates: Logistic Regression, Random Forest, RBF SVM, Gradient-Boosted Trees, and experimental 1D CNN.
- Gates: macro F1 ≥ 0.85; Walking and Other/Unknown recall ≥ 0.75.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-m', 'ml.train',
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--data-source-ref', KAGGLE_DATASET_REF,
    '--source-revision', RESOLVED_SOURCE_REVISION,
    '--wandb-mode', 'online',
    '--wandb-project', WANDB_PROJECT,
    '--wandb-group', f'wasp-loso-seed-{SEED}',
]
completed = subprocess.run(command, cwd=REPO_DIR, check=True, text=True, capture_output=True)
print(completed.stdout)


## Results

The report and model card deliberately retain the scope limitation. A passing result means only that this procedure met its cow-grouped public-dataset gates—not that it is field-ready.

In [ ]:
import json

report = json.loads((OUTPUT_DIR / 'benchmark_report.json').read_text(encoding='utf-8'))
selected = report['selected_model']
summary = {
    'selected_model': selected['selected_model_name'],
    'mean_macro_f1': selected['mean_macro_f1'],
    'mean_walking_recall': selected['mean_walking_recall'],
    'mean_unknown_recall': selected['mean_unknown_recall'],
    'release_gate_passed': selected['release_gate_passed'],
}
summary


## Next steps

- Download only aggregate reports and the selected model artifact for review.
- Do not treat this result as target-collar validation.
- Keep Phase 4 baseline learning and Phase 5 health-risk research disabled until compatible real longitudinal or veterinary-labelled data is available.